# 365 Probabilidades · Dia #071
## Qual a probabilidade de o tempo que você passa com seu filho mudar algo?

**Tipo:** Comportamental
**Data de publicação:** 2026-08-23
**Ferramenta:** Python
**Decisão analisada:** Mais horas juntos mudam o resultado?
**Hashtag:** #365Probabilidades #Dia071

---

### 📖 A História

Existe uma conta que quase toda mãe faz de cabeça, sem nunca ter escrito num papel.

Quantas horas eu passei com ele esta semana. Quantas eu deveria ter passado. A
diferença entre as duas, que ninguém cobra em voz alta e que ela cobra de si mesma
todo dia.

Essa conta pressupõe uma coisa que parece óbvia demais para ser testada: que **mais
horas produzem melhor resultado**. É a premissa inteira da maternidade intensiva, a
ideia de que o tempo materno é insubstituível e proporcional.

Só que premissa óbvia é justamente o tipo de coisa que nunca é medida. E quando
alguém finalmente mediu, o resultado não foi o esperado.

---

### 📚 O Conceito: dois tipos de tempo, e um nulo que precisa de cuidado

O estudo separa duas coisas que a gente costuma somar.

O **tempo engajado** é aquele em que mãe e filho estão fazendo algo juntos: conversar,
jogar, cozinhar, levar ao treino.

O **tempo acessível** é aquele em que a mãe está por perto e disponível, mas cada um
na sua atividade. Trabalhar na sala enquanto o filho estuda no quarto conta aqui.

A medida não vem de perguntar "quantas horas você acha que passa com seu filho", que
seria autorrelato de memória. Vem de **diário de tempo**: a pessoa registra o que
estava fazendo em cada intervalo do dia.

E aqui entra o cuidado que este projeto já fixou no #068: **resultado nulo não prova
ausência de efeito.** Ele limita o tamanho do efeito que teria passado despercebido.
Um estudo que não encontra relação pode significar que não há relação, ou que a
relação é menor do que aquele desenho conseguia enxergar.

Por isso a assinatura estatística deste dia é o efeito mínimo detectável, e não o
p-valor.

---

### 🧮 O Modelo

Diários de tempo de um painel longitudinal americano, com desfechos medidos por
escala e por teste.

**Fontes:**
- Milkie, M. A., Nomaguchi, K. M. & Denny, K. E., 2015 · "Does the Amount of Time
  Mothers Spend With Children or Adolescents Matter?" · *Journal of Marriage and
  Family* 77(2), 355-372 · Panel Study of Income Dynamics, Child Development
  Supplement · crianças de 3 a 11 anos, **N=1.605**, e adolescentes de 12 a 18 anos,
  **N=778** · tempo **engajado** e tempo **acessível** medidos por diário ·
  desfechos comportamentais, emocionais, acadêmicos e, na adolescência, comportamento
  de risco
- **Achado na infância:** a quantidade de tempo materno **não** se relacionou a
  comportamento, emoções ou desempenho acadêmico. Fatores de status social se
  mostraram importantes.
- **Achado na adolescência:** mais tempo materno **engajado** se relacionou a **menos
  comportamentos delinquentes**, e o tempo engajado com os dois pais juntos se
  relacionou a melhores resultados.
- **Debate formal na mesma revista:** Waldfogel (2016) e Kalil, Mayer e colegas
  publicaram comentários questionando o desenho e a interpretação, com réplica dos
  autores. Há também crítica pública apontando que a cobertura de imprensa concluiu
  mais do que o estudo sustenta.

**Precisão obrigatória:** o estudo mede **quantidade**, em horas. Nada nele diz que o
que acontece dentro dessas horas é irrelevante. A manchete de que "tempo com filho não
importa" é leitura que o próprio artigo não sustenta.

**Causalidade reversa, levantada pelos próprios autores:** mães de crianças com
dificuldades emocionais, comportamentais ou escolares podem ter escolhido passar mais
tempo com elas. Isso pode explicar parte da associação fraca, e a seta apontaria para
trás.

**Nota metodológica sobre o fator ×0.80:** não se aplica. O tempo vem de diário de
registro, não de estimativa de memória, e os desfechos vêm de escalas validadas e
testes de desempenho.


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("Bibliotecas carregadas")

Bibliotecas carregadas


In [6]:
# --- DADOS DA LITERATURA ---
# Milkie, Nomaguchi & Denny, 2015, Journal of Marriage and Family 77(2), 355-372
# Panel Study of Income Dynamics - Child Development Supplement

n_criancas    = 1_605      # 3 a 11 anos
n_adolescentes = 778       # 12 a 18 anos

# Achados, por faixa etaria e desfecho
# True  = houve associacao significativa
# False = nao houve
ACHADOS = {
    'Crianças (3 a 11)': {
        'Comportamento':  False,
        'Emoções':        False,
        'Desempenho academico': False,
    },
    'Adolescentes (12 a 18)': {
        'Comportamento delinquente (tempo engajado da mae)': True,
        'Resultados com os dois pais juntos':               True,
        'Demais desfechos':                                 False,
    },
}

# O que APARECEU como importante na infancia
fatores_relevantes = ['renda familiar', 'escolaridade materna', 'estrutura familiar']

aplica_fator_080 = False

print("=" * 70)
print("  DADOS - QUANTIDADE DE TEMPO MATERNO E DESFECHOS DO FILHO")
print("=" * 70)
print(f"\n  Milkie, Nomaguchi & Denny, 2015 (PSID-CDS):")
print(f"  -> Criancas de 3 a 11 anos:      N = {n_criancas:,}".replace(",", "."))
print(f"  -> Adolescentes de 12 a 18 anos: N = {n_adolescentes}")
print(f"  -> Tempo medido por DIARIO, nao por estimativa de memoria")
print(f"  -> Dois tipos: tempo ENGAJADO e tempo ACESSIVEL")
for faixa, desfechos in ACHADOS.items():
    print(f"\n  {faixa}:")
    for nome, houve in desfechos.items():
        marca = "ASSOCIACAO" if houve else "sem associacao"
        print(f"  -> {nome:<52} {marca}")
print(f"\n  O que apareceu como importante na infancia:")
for f in fatores_relevantes:
    print(f"  -> {f}")
print(f"\n  Fator x0.80 aplicado: {aplica_fator_080}")
print("=" * 70)

  DADOS - QUANTIDADE DE TEMPO MATERNO E DESFECHOS DO FILHO

  Milkie, Nomaguchi & Denny, 2015 (PSID-CDS):
  -> Criancas de 3 a 11 anos:      N = 1.605
  -> Adolescentes de 12 a 18 anos: N = 778
  -> Tempo medido por DIARIO, nao por estimativa de memoria
  -> Dois tipos: tempo ENGAJADO e tempo ACESSIVEL

  Crianças (3 a 11):
  -> Comportamento                                        sem associacao
  -> Emoções                                              sem associacao
  -> Desempenho academico                                 sem associacao

  Adolescentes (12 a 18):
  -> Comportamento delinquente (tempo engajado da mae)    ASSOCIACAO
  -> Resultados com os dois pais juntos                   ASSOCIACAO
  -> Demais desfechos                                     sem associacao

  O que apareceu como importante na infancia:
  -> renda familiar
  -> escolaridade materna
  -> estrutura familiar

  Fator x0.80 aplicado: False


In [7]:
# --- O MODELO ---
# O achado da infancia e um NULO. Nulo nao prova ausencia: limita o tamanho.
# Assinatura: efeito minimo detectavel, na escala de correlacao.

ALFA = 0.05
PODER = 0.80


def poder_correlacao(r, n, alfa=ALFA):
    "Poder para detectar uma correlacao r com n observacoes (transformacao z de Fisher)."
    if r <= 0:
        return alfa
    z = 0.5 * np.log((1 + r) / (1 - r))          # z de Fisher
    ep = 1 / np.sqrt(n - 3)
    critico = stats.norm.ppf(1 - alfa / 2)
    poder = stats.norm.sf(critico - abs(z) / ep) + stats.norm.cdf(-critico - abs(z) / ep)
    return float(np.nan_to_num(poder, nan=0.0))


def r_minimo(n, alfa=ALFA, poder=PODER):
    return optimize.brentq(lambda r: poder_correlacao(r, n, alfa) - poder, 1e-4, 0.99)


r_min_criancas = r_minimo(n_criancas)
r_min_adolesc = r_minimo(n_adolescentes)

# Quanto da variacao do desfecho isso representa
var_explicada_criancas = r_min_criancas ** 2 * 100
var_explicada_adolesc = r_min_adolesc ** 2 * 100

# Traducao: com esse N, qual correlacao passaria despercebida?
convencionais = {'muito pequena (r=0,05)': 0.05, 'pequena (r=0,10)': 0.10,
                 'moderada (r=0,20)': 0.20, 'grande (r=0,30)': 0.30}

print("=" * 70)
print("  MODELO - O QUE ESTE ESTUDO CONSEGUIRIA ENXERGAR")
print("=" * 70)
print(f"\n  Efeito minimo detectavel, com {PODER*100:.0f}% de poder:")
print(f"  -> Criancas (N={n_criancas:,})".replace(",", ".") +
      f":     r = {r_min_criancas:.3f}"
      f"  ({var_explicada_criancas:.2f}% da variacao)")
print(f"  -> Adolescentes (N={n_adolescentes}): r = {r_min_adolesc:.3f}"
      f"  ({var_explicada_adolesc:.2f}% da variacao)")
print(f"\n  Poder do estudo na amostra infantil, por tamanho de efeito:")
for nome, r in convencionais.items():
    print(f"  -> {nome:<24} poder = {poder_correlacao(r, n_criancas)*100:.0f}%")
print(f"\n  Poder na amostra adolescente:")
for nome, r in convencionais.items():
    print(f"  -> {nome:<24} poder = {poder_correlacao(r, n_adolescentes)*100:.0f}%")
print(f"\n  LEITURA CORRETA DO NULO:")
print(f"  -> A amostra infantil enxergaria qualquer correlacao acima de"
      f" {r_min_criancas:.2f}.")
print(f"  -> Como nao viu, o efeito da QUANTIDADE de horas, se existe,")
print(f"     explica menos de {var_explicada_criancas:.1f}% da variacao dos desfechos.")
print(f"  -> Isso NAO e o mesmo que dizer que o tempo nao importa.")
print(f"     E dizer que a CONTAGEM DE HORAS nao e a variavel que explica.")
print(f"\n  E na adolescencia, com amostra MENOR e portanto menos poder,")
print(f"  a associacao com tempo engajado apareceu mesmo assim.")
print("=" * 70)

  MODELO - O QUE ESTE ESTUDO CONSEGUIRIA ENXERGAR

  Efeito minimo detectavel, com 80% de poder:
  -> Criancas (N=1.605):     r = 0.070  (0.49% da variacao)
  -> Adolescentes (N=778): r = 0.100  (1.01% da variacao)

  Poder do estudo na amostra infantil, por tamanho de efeito:
  -> muito pequena (r=0,05)   poder = 52%
  -> pequena (r=0,10)         poder = 98%
  -> moderada (r=0,20)        poder = 100%
  -> grande (r=0,30)          poder = 100%

  Poder na amostra adolescente:
  -> muito pequena (r=0,05)   poder = 29%
  -> pequena (r=0,10)         poder = 80%
  -> moderada (r=0,20)        poder = 100%
  -> grande (r=0,30)          poder = 100%

  LEITURA CORRETA DO NULO:
  -> A amostra infantil enxergaria qualquer correlacao acima de 0.07.
  -> Como nao viu, o efeito da QUANTIDADE de horas, se existe,
     explica menos de 0.5% da variacao dos desfechos.
  -> Isso NAO e o mesmo que dizer que o tempo nao importa.
     E dizer que a CONTAGEM DE HORAS nao e a variavel que explica.

  E na 

In [8]:
# --- VISUALIZACAO ---

def br(n):
    return f"{n:,}".replace(",", ".")


DOURADO = '#c8a84b'
VERMELHO = '#c0392b'
VERDE = '#2a8a82'
CINZA = '#6b6a64'

# GRAFICO 1 - Onde apareceu e onde nao apareceu
fig1, ax1 = plt.subplots(figsize=(12, 8))

linhas = []
for faixa, desfechos in ACHADOS.items():
    for nome, houve in desfechos.items():
        linhas.append((faixa, nome, houve))

y = np.arange(len(linhas))[::-1]
for yi, (faixa, nome, houve) in zip(y, linhas):
    cor = VERDE if houve else CINZA
    largura = 1.0 if houve else 0.12
    ax1.barh([yi], [largura], color=cor, alpha=0.85, height=0.55)
    texto = 'associação' if houve else 'sem associação'
    ax1.text(largura + 0.03, yi, texto, va='center', fontsize=13,
             color=cor, fontweight='bold')

ax1.set_yticks(y)
ax1.set_yticklabels([f'{n}\n({f})' for f, n, _ in linhas], fontsize=10)
ax1.set_xlim(0, 1.75)
ax1.set_xticks([])
ax1.grid(False)
ax1.set_title('A quantidade de horas só apareceu na adolescência\n'
              f'Milkie, Nomaguchi & Denny, 2015 · N={br(n_criancas)} crianças e '
              f'N={n_adolescentes} adolescentes',
              fontsize=14, pad=18)
ax1.text(0.5, -0.09,
         'Na infância, o que apareceu como importante foram fatores de status social, '
         'não horas.',
         transform=ax1.transAxes, ha='center', fontsize=12, color=CINZA, style='italic')

plt.figtext(0.5, 0.005,
            'Fonte: Milkie, Nomaguchi & Denny, 2015, Journal of Marriage and Family 77(2)'
            '  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-071-grafico-01-onde-apareceu.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 1 salvo")

# GRAFICO 2 - Assinatura: o efeito minimo detectavel
fig2, ax2 = plt.subplots(figsize=(12, 8))

rs = np.linspace(0.01, 0.35, 400)
ax2.plot(rs, [poder_correlacao(r, n_criancas) * 100 for r in rs],
         color=DOURADO, linewidth=3, label=f'Crianças (N={br(n_criancas)})')
ax2.plot(rs, [poder_correlacao(r, n_adolescentes) * 100 for r in rs],
         color=VERDE, linewidth=3, label=f'Adolescentes (N={n_adolescentes})')

ax2.axhline(y=80, color=CINZA, linestyle='--', linewidth=1.8)
ax2.axvline(x=r_min_criancas, color=DOURADO, linestyle=':', linewidth=2)
ax2.axvline(x=r_min_adolesc, color=VERDE, linestyle=':', linewidth=2)

ax2.text(r_min_criancas + 0.004, 20, f'r = {r_min_criancas:.2f}'.replace('.', ','),
         fontsize=13, color=DOURADO, fontweight='bold')
ax2.text(r_min_adolesc + 0.004, 34, f'r = {r_min_adolesc:.2f}'.replace('.', ','),
         fontsize=13, color=VERDE, fontweight='bold')
ax2.text(0.345, 82, 'poder de 80%', ha='right', fontsize=11, color=CINZA)

ax2.set_xlim(0, 0.35)
ax2.set_ylim(0, 106)
ax2.set_xlabel('Correlação entre horas com a mãe e desfecho do filho')
ax2.set_ylabel('Chance de o estudo enxergar essa correlação (%)')
ax2.legend(frameon=False, fontsize=13, loc='lower right')
ax2.set_title('A assinatura estatística: o que este estudo conseguiria ver\n'
              'Um resultado nulo não prova ausência. Ele limita o tamanho.',
              fontsize=14, pad=18)
ax2.text(0.5, -0.135,
         'A amostra infantil enxergaria qualquer correlação acima de '
         + f'{r_min_criancas:.2f}'.replace('.', ',')
         + '. Como não viu, a quantidade de horas explica menos de '
         + f'{r_min_criancas**2*100:.1f}'.replace('.', ',') + '% da variação.',
         transform=ax2.transAxes, ha='center', fontsize=12, color=DOURADO,
         fontweight='bold')

plt.figtext(0.5, 0.005,
            'Cálculo de poder por transformação z de Fisher, alfa = 0,05'
            '  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-071-grafico-02-poder.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 2 salvo")

# GRAFICO 3 - O paradoxo: menos gente, e mesmo assim apareceu
fig3, ax3 = plt.subplots(figsize=(12, 7))

barras = ax3.bar(['Crianças\n3 a 11 anos', 'Adolescentes\n12 a 18 anos'],
                 [n_criancas, n_adolescentes],
                 color=[CINZA, VERDE], alpha=0.85, width=0.45)
for barra, valor in zip(barras, [n_criancas, n_adolescentes]):
    ax3.text(barra.get_x() + barra.get_width() / 2, valor + 40,
             f'N = {br(valor)}', ha='center', fontsize=19,
             fontweight='bold')

ax3.text(0, n_criancas / 2, 'nenhuma associação\ncom quantidade de horas',
         ha='center', va='center', fontsize=14, color='white', fontweight='bold')
ax3.text(1, n_adolescentes / 2, 'associação com\ntempo engajado',
         ha='center', va='center', fontsize=14, color='white', fontweight='bold')

ax3.set_ylim(0, n_criancas * 1.2)
ax3.set_ylabel('Tamanho da amostra')
ax3.set_title('O detalhe que quase ninguém nota\n'
              'A amostra adolescente é menos da metade da infantil, '
              'e mesmo assim o efeito apareceu ali',
              fontsize=14, pad=18)
ax3.text(0.5, -0.16,
         'Amostra menor significa menos poder estatístico. Encontrar associação com '
         'menos gente\ntorna o achado adolescente mais notável, não menos.',
         transform=ax3.transAxes, ha='center', fontsize=11, color=CINZA, style='italic')

plt.figtext(0.5, 0.005,
            'Fonte: Milkie, Nomaguchi & Denny, 2015  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-071-grafico-03-amostras.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 3 salvo")

Grafico 1 salvo
Grafico 2 salvo
Grafico 3 salvo


### 💡 O Insight

Mil seiscentas e cinco crianças de 3 a 11 anos. Setecentos e setenta e oito
adolescentes de 12 a 18. Tempo medido por diário, não por estimativa de memória.

**Na infância, a quantidade de horas com a mãe não se relacionou a comportamento, a
emoções nem a desempenho escolar.** O que apareceu como importante foram fatores de
status social: renda, escolaridade, estrutura familiar.

Antes de você respirar aliviada ou ficar irritada, uma pausa. Esse é um resultado
nulo, e resultado nulo é onde mais se exagera.

Calculei o que essa amostra conseguiria enxergar. Com 1.605 crianças, qualquer
correlação acima de aproximadamente 0,07 apareceria. Como não apareceu, a leitura
correta é: **se a quantidade de horas tem efeito, ele explica menos de meio por cento
da variação dos desfechos.**

Isso não é o mesmo que dizer que o tempo não importa. É dizer que **a contagem de
horas não é a variável que explica**. São coisas diferentes, e a imprensa da época
misturou as duas, o que gerou comentários formais de outros pesquisadores na própria
revista.

Agora a parte que quase ninguém contou.

Na adolescência, o resultado **muda de direção**. Mais tempo materno engajado se
relacionou a menos comportamento delinquente. E o tempo com os dois pais juntos se
relacionou a melhores resultados.

Repara no detalhe estatístico: a amostra adolescente é **menos da metade** da
infantil. Amostra menor enxerga menos. Encontrar associação com menos gente torna
esse achado mais notável, não menos.

E os próprios autores apontam a ironia: quase toda a pressão cultural sobre mães se
concentra na presença junto de crianças pequenas, e quase nenhuma na adolescência,
que pode ser justamente o estágio em que o tempo junto tem mais efeito.

Fica então uma reorganização, não uma absolvição.

A conta de horas com uma criança de cinco anos, aquela que tira o sono de tanta gente,
não é onde o dado aponta. A conversa com um adolescente de quinze, que costuma parecer
menos urgente porque ele já se vira sozinho, é onde ele aponta.

*Em que fase você mais se cobrou presença, e em que fase você mais foi cobrada?*

---

### ⚠️ Limitações do Modelo

- **O estudo mede quantidade, em horas.** Nada nele avalia o que acontece dentro
  dessas horas. Concluir que "tempo com filho não importa" é leitura que o próprio
  artigo não sustenta.
- **Causalidade reversa, levantada pelos autores.** Mães de crianças com dificuldades
  podem ter escolhido passar mais tempo com elas, o que enfraqueceria a associação
  observada. A seta pode apontar para trás.
- **É observacional.** Ninguém sorteou quantas horas cada mãe passaria com o filho.
- **O achado gerou debate formal na mesma revista.** Waldfogel (2016) e Kalil, Mayer e
  colegas publicaram comentários questionando desenho e interpretação, com réplica dos
  autores. Há também crítica pública de que a cobertura de imprensa concluiu mais do
  que o estudo permite. O debate segue aberto.
- **O efeito mínimo detectável deste notebook é uma aproximação.** Foi calculado na
  escala de correlação simples, por transformação z de Fisher, enquanto o artigo usa
  modelos de regressão com covariáveis. Serve como ordem de grandeza do que o desenho
  enxergaria, não como reprodução da análise original.
- Os coeficientes exatos do achado adolescente não foram reproduzidos aqui. O notebook
  registra direção e significância, conforme publicado.
- Amostra americana, de um painel longitudinal específico. Generalização para o Brasil
  não está testada.
- Fator ×0.80 não aplicado: tempo por diário de registro e desfechos por escala
  validada e teste de desempenho.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
